# A2组（G21）实验：2.1 插值与Runge现象、2.2 最小二乘与切比雪夫逼近

本文件只用于 2.1、2.2 两节，与 `A2组-实验.ipynb`（2.3、2.4）分工，互不覆盖。

- 依赖：NumPy、SciPy、Matplotlib。**不导入 torch**，另依赖threadpoolctl控制线程；安装下方环境文件后可运行。
- 用法：`Kernel → Restart Kernel and Run All Cells`，等全部单元出现执行编号后 `Ctrl + S` 保存。
- 本文件的数字与 `2.1-2.2-实测摘要.txt` 一致；图和正文 `图片/A2组-*.png` 由本文件重新生成。

预测作为补写的教学预期，不作为原始实验运行前已登记的记录。


In [1]:
import os
for _key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_key] = "1"
import sys, platform
import numpy as np, scipy
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
_thread_limit = threadpool_limits(limits=1)
from pathlib import Path
from scipy.interpolate import BarycentricInterpolator

IMG = Path("图片"); IMG.mkdir(exist_ok=True)
print({"python": platform.python_version(), "numpy": np.__version__,
       "scipy": scipy.__version__, "matplotlib": matplotlib.__version__})

{'python': '3.12.10', 'numpy': '1.26.4', 'scipy': '1.14.1', 'matplotlib': '3.9.2'}


## 2.1 插值与Runge现象

### 实验2.1-A：同样节点数，两种位置

对 $f(x)=1/(1+25x^2)$ 在 $[-1,1]$ 上取 $n=5,9,11,15,21,31$ 两组节点（等距、Chebyshev），
在4001点等距网格上比较区间最大绝对误差。预测：等距一组不随 $n$ 单调下降。


In [2]:
def f(x):
    return 1.0 / (1.0 + 25.0 * x * x)

def nodes_equi(n):
    return np.linspace(-1.0, 1.0, n)

def nodes_cheb(n):
    xc = np.cos(np.pi * (2 * np.arange(n) + 1) / (2 * n))
    return np.sort(xc)

grid = np.linspace(-1.0, 1.0, 4001)
rows = []
print(f"{'n':>4}{'equidistant':>16}{'chebyshev':>14}{'ratio':>12}")
for n in (5, 9, 11, 15, 21, 31):
    xe, xc = nodes_equi(n), nodes_cheb(n)
    ee = float(np.max(np.abs(BarycentricInterpolator(xe, f(xe), random_state=2026)(grid) - f(grid))))
    ec = float(np.max(np.abs(BarycentricInterpolator(xc, f(xc), random_state=2026)(grid) - f(grid))))
    rows.append((n, ee, ec, ee / ec))
    print(f"{n:>4}{ee:>16.6e}{ec:>14.6e}{ee / ec:>12.4g}")

assert 1.90 < rows[2][1] < 1.93, "等距 n=11 的误差与正文不符"
assert 0.10 < rows[2][2] < 0.12, "Chebyshev n=11 的误差与正文不符"
assert rows[-1][1] > 1e3, "等距 n=31 应远大于 1e3"
assert rows[-1][2] < 5e-3, "Chebyshev n=31 应小于 5e-3"
print("节点加密不降反升：n=11 -> n=31 的等距误差 %.3g -> %.3g" % (rows[2][1], rows[-1][1]))

   n     equidistant     chebyshev       ratio
   5    4.383566e-01  4.020167e-01        1.09
   9    1.045174e+00  1.708356e-01       6.118
  11    1.915643e+00  1.091533e-01       17.55
  15    7.194881e+00  4.660207e-02       154.4
  21    5.982231e+01  1.533373e-02        3901
  31    2.387998e+03  2.061544e-03   1.158e+06
节点加密不降反升：n=11 -> n=31 的等距误差 1.92 -> 2.39e+03


### 实验2.1-B：误差在哪里最大

节点上误差应为0，误差峰值出现在端点附近。取 $n=11$，定位区间最大误差位置，
并比较 $x=0.95$ 处两种插值与真值。图同时保存到 `图片/A2组-Runge现象.png`。


In [3]:
n = 11
xe, xc = nodes_equi(n), nodes_cheb(n)
pe = BarycentricInterpolator(xe, f(xe), random_state=2026)
pc = BarycentricInterpolator(xc, f(xc), random_state=2026)

print(f"节点处最大误差（等距）  = {np.max(np.abs(pe(xe) - f(xe))):.1e}")
g = np.linspace(-1.0, 1.0, 2001)
err = np.abs(pe(g) - f(g))
print(f"区间最大误差位置        = {abs(g[np.argmax(err)]):.4f}（及其对称点）")
print(f"x=0.95 等距             = {float(pe(0.95)):.4f}")
print(f"x=0.95 Chebyshev        = {float(pc(0.95)):.4f}")
print(f"x=0.95 真值             = {float(f(0.95)):.4f}")

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(g, f(g), "k-", lw=2.2, label="true f(x)=1/(1+25x^2)")
ax.plot(g, pe(g), "C3--", lw=1.3, label="equispaced n=11")
ax.plot(g, pc(g), "C0-.", lw=1.3, label="Chebyshev n=11")
ax.plot(xe, f(xe), "C3o", ms=5, label="equispaced nodes")
ax.plot(xc, f(xc), "C0s", ms=4, mfc="none", label="Chebyshev nodes")
ax.axvline(0.95, color="gray", ls=":", lw=1)
ax.set_xlim(-1, 1); ax.set_ylim(-0.45, 2.6)
ax.set_xlabel("x"); ax.set_ylabel("value")
ax.set_title("Runge phenomenon: 11 points, two node choices")
ax.grid(alpha=0.3); ax.legend(fontsize=7.5, loc="upper center", ncol=2, framealpha=0.95)
fig.tight_layout()
fig.savefig(IMG / "A2组-Runge现象.png", dpi=150)
plt.close(fig)

assert abs(float(pe(0.95)) - 1.9236) < 5e-4, "x=0.95 的插值结果与正文不符"
assert abs(float(f(0.95)) - 0.0424) < 5e-4

节点处最大误差（等距）  = 0.0e+00
区间最大误差位置        = 0.9400（及其对称点）
x=0.95 等距             = 1.9236
x=0.95 Chebyshev        = 0.0855
x=0.95 真值             = 0.0424


## 2.2 最小二乘与切比雪夫逼近

### 实验2.2-A：同一个拟合，两种基

训练集：$e^x$ 在 $[0,1]$ 上15个等距点加噪声 $\mathcal{N}(0,0.03^2)$，种子2026；
测试集：同区间400点解析值。比较次数 $d=3,6,9,12$ 下幂基与Chebyshev基的条件数、训练残差与测试误差。
预测：两个解几乎相同，条件数差很多。


In [4]:
rng = np.random.default_rng(2026)
m = 15
x = np.linspace(0.0, 1.0, m)
y = np.exp(x) + rng.normal(0.0, 0.03, m)
xt = np.linspace(0.0, 1.0, 400)

print(f"{'d':>3}{'cond_mono':>13}{'train_mono':>12}{'test_mono':>12}"
      f"{'cond_cheb':>12}{'train_cheb':>12}{'test_cheb':>12}")
caps = {}
for d in (3, 6, 9, 12):
    V = np.vander(x, d + 1)
    c, *_ = np.linalg.lstsq(V, y, rcond=None)
    tr = float(np.max(np.abs(V @ c - y)))
    te = float(np.max(np.abs(np.vander(xt, d + 1) @ c - np.exp(xt))))

    T = np.polynomial.chebyshev.chebvander(2 * x - 1, d)
    cc, *_ = np.linalg.lstsq(T, y, rcond=None)
    trc = float(np.max(np.abs(T @ cc - y)))
    tec = float(np.max(np.abs(
        np.polynomial.chebyshev.chebvander(2 * xt - 1, d) @ cc - np.exp(xt))))

    caps[d] = (np.linalg.cond(V), np.linalg.cond(T))
    print(f"{d:>3}{caps[d][0]:>13.2e}{tr:>12.3e}{te:>12.3e}"
          f"{caps[d][1]:>12.2e}{trc:>12.3e}{tec:>12.3e}")

assert 2.2e9 < caps[12][0] < 2.6e9, "d=12 幂基条件数与正文不符"
assert 15 < caps[12][1] < 18, "d=12 Chebyshev 基条件数与正文不符"

# 正规方程与直接最小二乘：条件数平方
d = 9
V = np.vander(x, d + 1)
c_ls, *_ = np.linalg.lstsq(V, y, rcond=None)
c_ne = np.linalg.solve(V.T @ V, V.T @ y)
print(f"cond(A^T A) = {np.linalg.cond(V.T @ V):.2e}")
print(f"cond(A)     = {np.linalg.cond(V):.2e}")
print(f"两种解法系数最大差 = {float(np.max(np.abs(c_ls - c_ne))):.3f}")

  d    cond_mono  train_mono   test_mono   cond_cheb  train_cheb   test_cheb
  3     1.05e+02   5.168e-02   2.045e-02    1.82e+00   5.168e-02   2.045e-02
  6     1.82e+04   5.203e-02   2.530e-02    1.85e+00   5.203e-02   2.530e-02
  9     4.63e+06   3.419e-02   3.150e-02    2.99e+00   3.419e-02   3.150e-02
 12     2.41e+09   5.639e-03   1.764e-01    1.65e+01   5.639e-03   1.764e-01
cond(A^T A) = 2.14e+13
cond(A)     = 4.63e+06
两种解法系数最大差 = 4.152


### 实验2.2-B：残差小，未必画得对

把次数加到14：15个数据点、15个未知系数，多项式穿过全部训练点。
比较训练残差与独立测试误差，并作图。预测：训练残差趋近0，测试误差反而变大。


In [5]:
print(f"{'d':>3}{'train_max_residual':>21}{'test_max_error':>17}")
res = {}
for d in (1, 2, 3, 6, 9, 12, 14):
    V = np.vander(x, d + 1)
    c, *_ = np.linalg.lstsq(V, y, rcond=None)
    tr = float(np.max(np.abs(V @ c - y)))
    te = float(np.max(np.abs(np.vander(xt, d + 1) @ c - np.exp(xt))))
    res[d] = (tr, te)
    print(f"{d:>3}{tr:>21.3e}{te:>17.3e}")

assert res[14][0] < 1e-5, "d=14 应几乎穿过所有训练点"
assert res[14][1] > 0.5, "d=14 的测试误差应与正文同量级"
print(f"d=14：训练残差 {res[14][0]:.3e}，测试误差 {res[14][1]:.3f}")

fig, axs = plt.subplots(1, 2, figsize=(9.8, 4.0))
deg = np.arange(1, 15)
cm = [np.linalg.cond(np.vander(x, d + 1)) for d in deg]
cc = [np.linalg.cond(np.polynomial.chebyshev.chebvander(2 * x - 1, d)) for d in deg]
axs[0].semilogy(deg, cm, "C3o-", ms=3.5, label="monomial basis")
axs[0].semilogy(deg, cc, "C0s-", ms=3.5, label="Chebyshev basis")
axs[0].set_xlabel("polynomial degree"); axs[0].set_ylabel("cond of design matrix")
axs[0].set_title("Same fit space, different conditioning")
axs[0].grid(alpha=0.3, which="both"); axs[0].legend(fontsize=8)

xf = np.linspace(0, 1, 600)
for d, color, ls, lab in ((3, "C0", "-", "degree 3"),
                          (14, "C3", "--", "degree 14")):
    co, *_ = np.linalg.lstsq(np.vander(x, d + 1), y, rcond=None)
    axs[1].plot(xf, np.vander(xf, d + 1) @ co, ls, color=color, lw=1.5, label=lab)
axs[1].plot(xf, np.exp(xf), "k-", lw=2.2, label="true exp(x)")
axs[1].plot(x, y, "ko", ms=4.5, label="15 noisy training points")
axs[1].set_xlabel("x"); axs[1].set_ylabel("y"); axs[1].set_ylim(-1.6, 4.2)
axs[1].set_title("Small training residual, poor prediction")
axs[1].grid(alpha=0.3); axs[1].legend(fontsize=7.5, loc="upper left")
fig.tight_layout()
fig.savefig(IMG / "A2组-基函数与过拟合.png", dpi=150)
plt.close(fig)

  d   train_max_residual   test_max_error
  1            1.386e-01        1.338e-01
  2            5.238e-02        1.366e-02
  3            5.168e-02        2.045e-02
  6            5.203e-02        2.530e-02
  9            3.419e-02        3.150e-02
 12            5.639e-03        1.764e-01
 14            1.028e-07        9.688e-01
d=14：训练残差 1.028e-07，测试误差 0.969


### 本节结论与边界

- 插值：节点上精确不代表区间内精确；本例所列等距节点加密时误差增大，Chebyshev节点只优化 $\omega$，不保证任意函数收敛。
- 最小二乘：条件数决定能不能算（$\kappa(A^{\mathsf T}A)=\kappa(A)^2$），基函数不改变拟合空间；残差小不代表泛化好。
- 全部指标为采样网格上的数值指标，不是连续区间上的严格上界；2.2评价网格与训练集共有8个重合点（含两个端点），其余392点不重合；2.1评价网格也包含部分插值节点。评价网格均不参与拟合。


## 本次执行证据

以下保存本次环境与关键数值，供正文和题解核对；不代替B/C独立复现。

In [6]:
import json
from threadpoolctl import threadpool_info
evidence = {
    "environment": {"python": platform.python_version(), "numpy": np.__version__,
                    "scipy": scipy.__version__, "matplotlib": matplotlib.__version__,
                    "seed": 2026, "dtype": "float64", "device": "CPU",
                    "blas_threads": [{"internal_api": t["internal_api"], "num_threads": t["num_threads"]} for t in threadpool_info()]},
    "interpolation": [{"n": int(a), "equidistant_max_error": b, "chebyshev_max_error": c, "ratio": d} for a,b,c,d in rows],
    "probe": {"x": 0.95, "equidistant": float(pe(0.95)), "chebyshev": float(pc(0.95)), "reference": float(f(0.95)),
              "peak_abs_x": float(abs(grid[np.argmax(np.abs(pe(grid)-f(grid)))]))},
    "conditioning": {str(d): [float(a),float(b)] for d,(a,b) in caps.items()},
    "normal_equation": {"condition": float(np.linalg.cond(np.vander(x,10).T@np.vander(x,10))),
                        "coefficient_max_difference": float(np.max(np.abs(c_ls-c_ne)))},
    "degree_errors": {str(d): {"train_max_residual": a,"test_max_error": b} for d,(a,b) in res.items()},
    "evaluation_overlap": {"training_points": 15,"evaluation_points": 400,"shared_endpoints": 2,
                          "shared_points": int(np.sum(np.min(np.abs(xt[:,None]-x[None,:]),axis=1)<1e-12)),
                          "remaining_points": int(np.sum(np.min(np.abs(xt[:,None]-x[None,:]),axis=1)>=1e-12))}
}
Path("数据").mkdir(exist_ok=True)
Path("数据/A2组-2.1-2.2-运行证据.json").write_text(json.dumps(evidence,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
print(json.dumps(evidence,ensure_ascii=False,indent=2))


{
  "environment": {
    "python": "3.12.10",
    "numpy": "1.26.4",
    "scipy": "1.14.1",
    "matplotlib": "3.9.2",
    "seed": 2026,
    "dtype": "float64",
    "device": "CPU",
    "blas_threads": [
      {
        "internal_api": "openblas",
        "num_threads": 1
      },
      {
        "internal_api": "openblas",
        "num_threads": 1
      }
    ]
  },
  "interpolation": [
    {
      "n": 5,
      "equidistant_max_error": 0.4383566395255679,
      "chebyshev_max_error": 0.40201674193790876,
      "ratio": 1.0903939905897546
    },
    {
      "n": 9,
      "equidistant_max_error": 1.045173911783697,
      "chebyshev_max_error": 0.1708355947318404,
      "ratio": 6.118010204046177
    },
    {
      "n": 11,
      "equidistant_max_error": 1.9156430502192483,
      "chebyshev_max_error": 0.10915326641231005,
      "ratio": 17.550029542709193
    },
    {
      "n": 15,
      "equidistant_max_error": 7.194881107233082,
      "chebyshev_max_error": 0.04660207211184919,
    